# GameTheory-14c — Jeux spatiaux (C#) : quand la coopération gagne en grappes

**Série GameTheory** · Jumeau C# du carnet [GameTheory-14c Python](GameTheory-14c-Jeux-Spatiaux-Cooperation-Python.ipynb)

[← Retour à la série](README.md) · Précédent : [GameTheory-14 — Jeux différentiels](GameTheory-14-DifferentialGames-CSharp.ipynb) · Jumeau Python : [14c (Python)](GameTheory-14c-Jeux-Spatiaux-Cooperation-Python.ipynb)

> **Objectif.** Rejouer en **C# .NET** le modèle de jeu spatial du jumeau Python : la coopération (TFT) perd en duel pur, mais **gagne quand elle est organisée en grappes**. On calcule les mêmes trajectoires et le même diagramme de phase — avec des tables au lieu de plots, l'honnêteté du BCL.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");

// Matrice de gains du dilemme du prisonnier itéré (T=5, R=3, P=1, S=0)
static int Gain(char a, char b) => (a, b) switch
{
    ('C', 'C') => 3,
    ('C', 'D') => 0,
    ('D', 'C') => 5,
    _ => 1
};

// Stratégies : fonction de l'historique des coups de l'ADVERSAIRE
var STRATEGIES = new Dictionary<string, Func<List<char>, char>>
{
    ["tft"]      = h => h.Count == 0 ? 'C' : h[^1],
    ["tft2"]     = h => (h.Count >= 2 && (h[^2] == 'D' || h[^1] == 'D')) ? 'D' : 'C',
    ["grudger"]  = h => h.Contains('D') ? 'D' : 'C',
    ["allC"]     = _ => 'C',
    ["allD"]     = _ => 'D',
};

// Contrôle 1 — dynamique de la part de TFT dans une population bien mélangée
List<double> DynamiqueMelangee(double partTft, int nbTours = 200, double tauxAdoption = 0.1)
{
    double x = partTft;
    var suite = new List<double> { x };
    for (int t = 0; t < nbTours; t++)
    {
        double pTft  = x * 3.0 + (1.0 - x) * 0.5;
        double pAlld = x * 3.0 + (1.0 - x) * 1.0;
        if (pTft > pAlld)      x = Math.Min(1.0, x + tauxAdoption * (1.0 - x));
        else if (pAlld > pTft) x = Math.Max(0.0, x - tauxAdoption * x);
        suite.Add(x);
    }
    return suite;
}

// Contrôle 2 — duel TFT vs TFT avec bruit de mutation p
double DuelBruit(double p, int nbTours = 200, int seed = 42)
{
    var rng = new Random(seed);
    char prevB = 'C', prevA = 'C';
    int coop = 0;
    for (int t = 0; t < nbTours; t++)
    {
        char aInt = t == 0 ? 'C' : prevB;
        char bInt = t == 0 ? 'C' : prevA;
        char aPlay = rng.NextDouble() < p ? 'D' : aInt;
        char bPlay = rng.NextDouble() < p ? 'D' : bInt;
        coop += (aPlay == 'C' ? 1 : 0) + (bPlay == 'C' ? 1 : 0);
        (prevA, prevB) = (aPlay, bPlay);
    }
    return coop / (2.0 * nbTours);
}

Console.WriteLine($"contrôle bien mélangé prêt — {STRATEGIES.Count} stratégies disponibles");

Runtime : .NET 10.0.15


contrôle bien mélangé prêt — 5 stratégies disponibles


## 1. Contrôle — la part de TFT, et le duel bruyé

Deux calculs de contrôle avant le modèle spatial, **identiques au jumeau Python** :

- **Dynamique bien mélangée** : dans une population homogène, le bien-être moyen ne fait que suivre la part de TFT — il ne peut pas décider seul entre « tout coopère » et « tous trahissent ». C'est le modèle qui est trop grossier.
- **Duel bruyé** : deux TFT s'entredéfient au moindre bruit de mutation — la coopération est **fragile** sans structure spatiale.

In [2]:
// (a) Bien-être moyen dans la population bien mélangée, selon la part initiale de TFT
Console.WriteLine("(a) Dynamique bien mélangée (part initiale de TFT -> part finale) :");
Console.WriteLine("  part init.   part finale");
foreach (var x0 in new[] { 0.10, 0.20, 0.30, 0.50 })
{
    var suite = DynamiqueMelangee(x0);
    double partFinale = suite[^1];
    Console.WriteLine($"  {x0,6:0.00}     {partFinale,10:0.0000}");
}

// (b) Taux de coopération dans un duel TFT vs TFT, selon le bruit de mutation
Console.WriteLine();
Console.WriteLine("(b) Duel TFT vs TFT (bruit de mutation -> taux de coopération) :");
Console.WriteLine("  bruit p    coopération");
foreach (var p in new[] { 0.0, 0.01, 0.02, 0.05 })
{
    double coop = DuelBruit(p);
    Console.WriteLine($"  {p,7:0.00}     {coop,12:0.000}");
}

(a) Dynamique bien mélangée (part initiale de TFT -> part finale) :


  part init.   part finale


    0.10         0.0000


    0.20         0.0000


    0.30         0.0000


    0.50         0.0000


... (2 outputs left out)

     0.00            1.000


     0.01            0.140


     0.02            0.130


     0.05            0.073


### Lecture du résultat

**Sans bruit**, quel que soit l'ensemencement initial — de 10 % à 50 % de coopérateurs — la part de tit-for-tat **tend vers zéro** (0.0000 partout). La cause est arithmétique : en population mélangée, un AllD qui rencontre un TFT y gagne 5 au premier round puis 1 (la riposte), soit 3 de moyenne — exactement son gain contre un AllD, **sans payer le premier round coopératif**. Le défauteur est toujours aussi riche que le coopérateur, et un peu plus : TFT ne peut pas s'y imposer.

**Avec bruit**, le duel tit-for-tat contre lui-même s'effondre : sans inversion il reste coopératif (1.000), dès p = 0.01 la part coopérative tombe à **0.140**, et à p = 0.05 elle n'est plus que **0.073**. Un coup inversé désynchronise les deux ripostes — chacun riposte à la faute de l'autre, la riposte fabrique sa riposte — et la **boule de neige de rancunes** dévore la coopération. (Les valeurs numériques diffèrent de celles du jumeau Python — 0.3575, 0.1800, 0.0150 — : même modèle et même graine 42, mais le générateur pseudo-aléatoire de .NET ne produit pas la même suite que celui de Python ; l'effondrement monotone, lui, est identique.) C'est ce point de départ, mesuré ici, que la structure spatiale va contester en §3.1.

## 2. Le modèle spatial

On place les agents sur un **tore** `n×n`. Chaque cellule :

1. joue le dilemme du prisonnier **avec ses 8 voisins** (son action est fonction de l'historique de chaque voisin) ;
2. **accumule** ses gains ;
3. **adopte** la stratégie du voisin le plus riche.

C'est exactement le modèle du jumeau Python, traduit en BCL : `int[,]` pour la grille, `Dictionary<(int,int,int),List<char>>` pour les historiques par direction, et `Random` pour le bruit.

In [3]:
// 8 directions du tore ; l'inverse de la direction d est 7-d (diagonales incluses)
var DIRS = new[] { (-1,-1),(-1,0),(-1,1), (0,-1),(0,1), (1,-1),(1,0),(1,1) };
var IDX = new Dictionary<string,int>();
int k = 0;
foreach (var nom in STRATEGIES.Keys) IDX[nom] = k++;
var FONCS = STRATEGIES.Values.ToArray();

// modulo TORE-safe : en C# (-1 % 20) = -1 (pas 19 comme en Python) -> on re-decale
static int Torus(int v, int n) => ((v % n) + n) % n;

int[,] UnTour(int[,] grille, Dictionary<(int,int,int),List<char>> hist, int n, double p, Random rng)
{
    // (1) chaque cellule joue ses 8 voisins
    var actions = new Dictionary<(int i,int j,int d),char>();
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++)
            for (int d=0; d<8; d++)
            {
                char a = FONCS[grille[i,j]](hist[(i,j,d)]);
                if (rng.NextDouble() < p) a = a=='C' ? 'D' : 'C';   // bruit de mutation
                actions[(i,j,d)] = a;
            }
    // (2) accumulation des gains par cellule
    var gains = new double[n,n];
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++)
        {
            double tot = 0;
            for (int d=0; d<8; d++)
            {
                int ni=Torus(i+DIRS[d].Item1, n), nj=Torus(j+DIRS[d].Item2, n);
                char opp = actions[(ni,nj,7-d)];
                tot += Gain(actions[(i,j,d)], opp);
            }
            gains[i,j] = tot;
        }
    // (3) adoption du voisin le plus riche
    var nouvelle = new int[n,n];
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++)
        {
            int meilleur = grille[i,j]; double meilleurG = -1;
            for (int d=0; d<8; d++)
            {
                int ni=Torus(i+DIRS[d].Item1, n), nj=Torus(j+DIRS[d].Item2, n);
                if (gains[ni,nj] > meilleurG) { meilleurG = gains[ni,nj]; meilleur = grille[ni,nj]; }
            }
            nouvelle[i,j] = meilleur;
        }
    // mise à jour des historiques : ce que chaque voisin a joué contre moi
    foreach (var key in hist.Keys)
    {
        int i=key.Item1, j=key.Item2, d=key.Item3;
        int ni=Torus(i+DIRS[d].Item1, n), nj=Torus(j+DIRS[d].Item2, n);
        hist[key].Add(actions[(ni,nj,7-d)]);
    }
    return nouvelle;
}

(int[,], List<double>) SimulationSpatiale(double partInit, int n=20, int nbTours=300, double p=0.0, int seed=42, string coop="tft", string defaut="allD")
{
    var rng = new Random(seed);
    var grille = new int[n,n];
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++) grille[i,j] = IDX[defaut];
    var cases = new List<(int,int)>();
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++) cases.Add((i,j));
    cases = cases.OrderBy(_ => rng.NextDouble()).Take((int)Math.Round(partInit*n*n)).ToList();
    foreach (var (i,j) in cases) grille[i,j] = IDX[coop];
    var hist = new Dictionary<(int,int,int),List<char>>();
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++)
            for (int d=0; d<8; d++) hist[(i,j,d)] = new List<char>();
    var suite = new List<double>();
    for (int t=0; t<nbTours; t++)
    {
        int cIdx = IDX[coop];
        suite.Add(grille.Cast<int>().Count(v => v == cIdx) / (double)(n*n));
        grille = UnTour(grille, hist, n, p, rng);
    }
    return (grille, suite);
}

(double, int[,]) ClusterKxK(int kk, int n=20, int nbTours=300, int seed=42, double p=0.0)
{
    var rng = new Random(seed);
    var grille = new int[n,n];
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++) grille[i,j] = IDX["allD"];
    int c = n/2;
    for (int di=0; di<kk; di++)
        for (int dj=0; dj<kk; dj++)
            grille[c - kk/2 + di, c - kk/2 + dj] = IDX["tft"];
    var hist = new Dictionary<(int,int,int),List<char>>();
    for (int i=0;i<n;i++)
        for (int j=0;j<n;j++)
            for (int d=0; d<8; d++) hist[(i,j,d)] = new List<char>();
    for (int t=0; t<nbTours; t++) grille = UnTour(grille, hist, n, p, rng);
    return (grille.Cast<int>().Count(v => v == IDX["tft"]) / (double)(n*n), grille);
}

Console.WriteLine($"moteur spatial prêt — {DIRS.Length} voisins par cellule");

moteur spatial prêt — 8 voisins par cellule


## 3. Le diagramme de phase

Deux ensemencements, deux destins — le **diagramme de phase** du système :

- **(a) TFT dispersés** : on plante une part `x₀` de TFT **aléatoirement** dans un champ de `allD`.
- **(b) Blocs compacts** : un carré `k×k` de TFT au **centre** du tore.

On lit la part finale de TFT. C'est le même calcul que le jumeau Python, rendu en **table** (le BCL ne trace pas ; une table est l'honnêteté d'un outil sans matplotlib).

In [4]:
// (a) TFT dispersés : part initiale aléatoire -> part finale
Console.WriteLine("(a) TFT DISPERSÉS (part initiale -> part finale de TFT) :");
Console.WriteLine("  x0 init.   part finale");
foreach (var x0 in new[] { 0.05, 0.10, 0.15, 0.20, 0.30 })
{
    var (_, suite) = SimulationSpatiale(x0, n:20);
    Console.WriteLine($"  {x0,6:0.00}     {suite[^1],10:0.000}");
}

// (b) blocs compacts : carré k x k au centre -> part finale
Console.WriteLine();
Console.WriteLine("(b) BLOCS COMPACTS (carré k x k au centre -> part finale de TFT) :");
Console.WriteLine("  k (côté)  part init.   part finale");
foreach (var kk in new[] { 2, 3, 4, 5, 6, 8, 10 })
{
    var (part, _) = ClusterKxK(kk, n:20);
    double partInit = Math.Pow(kk/20.0, 2);
    Console.WriteLine($"  {kk,8}     {partInit,10:0.000}     {part,10:0.000}");
}

(a) TFT DISPERSÉS (part initiale -> part finale de TFT) :


  x0 init.   part finale


    0.05          0.000


    0.10          0.000


    0.15          0.000


    0.20          0.000


... (6 outputs left out)

         5          0.063          0.650


         6          0.090          0.750


         8          0.160          0.950


        10          0.250          1.000


### Lecture — le même ensemencement, deux destins

**La géométrie décide.** Un petit pourcentage de TFT **dispersé** s'éteint (chaque coopérateur isolé est mangé par ses `allD` voisins), tandis qu'un **bloc compact** du même pourcentage **survit et grignote** : à l'intérieur du bloc, tout le monde coopère et gagne 3 ; seul le **contour** perd, et la perte de contour croît en `k` (périmètre) plus lentement que le gain de surface (`k²`). La coopération n'est pas une question de **quantité** initiale, mais de **structure**.

### 3.1 Le bruit — et la « boule de neige » retournée

On ajoute un **bruit de mutation** `p` au bloc compact de taille 5. Chaque coup a une probabilité `p` d'être joué à l'envers. On mesure la part finale de TFT.

In [5]:
// Bloc compact 5x5, bruit de mutation croissant
Console.WriteLine("Bloc compact 5x5 — bruit de mutation -> part finale de TFT :");
Console.WriteLine("  bruit p    part finale");
foreach (var p in new[] { 0.0, 0.01, 0.02, 0.05 })
{
    var (part, _) = ClusterKxK(5, n:20, p:p);
    Console.WriteLine($"  {p,7:0.00}     {part,10:0.000}");
}

Bloc compact 5x5 — bruit de mutation -> part finale de TFT :


  bruit p    part finale


     0.00          0.650


     0.01          1.000


     0.02          1.000


     0.05          1.000


### Lecture — la « boule de neige », retournée

En duel pur, un seul bruit **détruit** la coopération (1.000 → 0.140, section 1). Dans le bloc spatial, le même bruit produit **l'effet inverse** : le bloc stable (0.650) ne fond pas, il **se propage** — la mutation permet aux coopérateurs robustes de franchir le contour et de conquérir le tore entier (0.650 → 1.000). La structure spatiale ne se contente pas d'absorber la mutation : elle en **fait un levier de diffusion**. C'est l'inverse exact du duel, où la même mutation est fatale.

## 4. Clusters — le seuil

Quel est le **plus petit** bloc qui survit ? On teste des carrés de côté `k = 1, 2, 3, 4` au centre du tore.

In [6]:
// Le plus petit bloc qui survit
Console.WriteLine("Carrés de côté k au centre — part finale de TFT :");
Console.WriteLine("  k (côté)   part finale   verdict");
foreach (var kk in new[] { 1, 2, 3, 4 })
{
    var (part, _) = ClusterKxK(kk, n:20);
    string verdict = part > 0.05 ? "survit" : "s'éteint";
    Console.WriteLine($"  {kk,8}     {part,10:0.000}     {verdict}");
}

Carrés de côté k au centre — part finale de TFT :


  k (côté)   part finale   verdict


         1          0.000     s'éteint


         2          0.000     s'éteint


         3          0.450     survit


         4          0.550     survit


### Lecture — le seuil de cluster

Il existe un **seuil de criticité** : en dessous, le bloc fond (la perte de contour dépasse le gain de surface) ; au-dessus, il s'installe et croît. C'est le même mécanisme **périmètre/surface** qui gouverne la nucléation physique — un pont concret entre théorie des jeux et condensation de la matière.

## 5. Synthèse

- Le **bien-être moyen** ne décide rien (section 1) : il faut un modèle à structure.
- En **duel**, la coopération est **fragile** au moindre bruit.
- En **espace**, elle devient **robuste** : les blocs compacts survivent, grignotent, et absorbent le bruit.
- Il existe un **seuil de criticité** de taille de cluster, gouverné par le rapport **périmètre/surface**.

La morale : **la coopération gagne quand elle est organisée**, pas quand elle est nombreuse.

## 6. Exercices

### Exercice 1 — une stratégie indulgente

Écris une stratégie **TFT indulgent** (`tit-for-tat` qui pardonne un trahi isolé) : elle trahit seulement si l'adversaire a trahi **deux fois de suite**. Complète le corps de la fonction.

In [7]:
// Exercice 1 — a completer : TFT indulgent (trahir seulement sur 2 trahisons d'affilée)
char TitForTatIndulgent(List<char> historique)
{
    // TODO etudiant : retourner 'D' si les 2 derniers coups de l'adversaire sont 'D',
    // sinon 'C' (et 'C' si l'historique est trop court).
    Console.WriteLine("Exercice a completer");
    return 'C';
}
Console.WriteLine("Exercice 1 : a completer");

Exercice 1 : a completer


### Exercice 2 — la part finale d'un bloc

Complète `PartFinaleCluster` pour qu'elle retourne la part finale de TFT d'un carré `k×k` au centre (réutilise `ClusterKxK`).

In [8]:
// Exercice 2 — a completer : part finale de TFT d'un carré k x k au centre
double PartFinaleCluster(int kk)
{
    // TODO etudiant : appeler ClusterKxK(kk) et retourner la part finale.
    Console.WriteLine("Exercice a completer");
    return 0.0;
}
Console.WriteLine("Exercice 2 : a completer");

Exercice 2 : a completer


### Exercice 3 — la trajectoire dans la population mélangée

Complète `PartCooperationTemps` pour qu'elle retourne la **suite temporelle** de la part de coopération d'une population bien mélangée démarrant à `x0` (réutilise `DynamiqueMelangee`).

In [9]:
// Exercice 3 — a completer : suite temporelle de la part de coopération (x0 -> ...)
List<double> PartCooperationTemps(double x0)
{
    // TODO etudiant : appeler DynamiqueMelangee(x0) et retourner la suite.
    Console.WriteLine("Exercice a completer");
    return new List<double>();
}
Console.WriteLine("Exercice 3 : a completer");

Exercice 3 : a completer


## Conclusion

Le jumeau C# reproduit fidèlement le modèle spatial du jumeau Python : **la coopération gagne quand elle est organisée en grappes**, avec un seuil de criticité gouverné par le rapport périmètre/surface. Tout est en BCL standard — `int[,]`, `Dictionary`, `Random`, `Linq` — sans dépendance externe. Les tables remplacent honnêtement les plots.

## Références

- Nowak, M. A. & May, R. M. (1992). *Evolutionary games and spatial chaos*. Nature, 359, 826-829. — le modèle de grille 2D de ce carnet.
- Nowak, M. A. (2006). *Five rules for the evolution of cooperation*. Science, 314(5805), 1560-1563. — la règle « structure spatiale » est l'une des cinq.
- Jumeau Python de ce carnet : [GameTheory-14c (Python)](GameTheory-14c-Jeux-Spatiaux-Cooperation-Python.ipynb).